# Fine-Tuning vs Prompt Engineering vs Base Model

This notebook demonstrates:
1. Creating a dataset
2. Fine-tuning a model
3. Comparing:
   - Base model
   - Prompt engineered model
   - Fine-tuned model

Goal: Help understand WHEN to use each approach.


## Setup

In [6]:
from openai import OpenAI
import json, time

client = OpenAI()


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

## Step 1: Create Training Data

In [ ]:
training_data = [
    {"messages": [
        {"role": "system", "content": "You explain concepts in a fun, simple, friendly way with examples."},
        {"role": "user", "content": "What is RAG?"},
        {"role": "assistant", "content": "RAG is like an open-book exam for AI! It first searches for info, then answers using it."}
    ]},
    {"messages": [
        {"role": "system", "content": "You explain concepts in a fun, simple, friendly way with examples."},
        {"role": "user", "content": "What are embeddings?"},
        {"role": "assistant", "content": "Embeddings are like turning sentences into coordinates on a map so AI can compare meaning."}
    ]}
]

with open("training.jsonl", "w") as f:
    for item in training_data:
        f.write(json.dumps(item) + "\n")

print("Training file created")


## Step 2: Upload Data

In [ ]:
file = client.files.create(
    file=open("training.jsonl", "rb"),
    purpose="fine-tune"
)
file_id = file.id
print("File ID:", file_id)


## Step 3: Fine-Tune Model

In [ ]:
job = client.fine_tuning.jobs.create(
    training_file=file_id,
    model="gpt-4o-mini"
)
print("Job ID:", job.id)


## Step 4: Monitor Training

In [ ]:
while True:
    status = client.fine_tuning.jobs.retrieve(job.id)
    print("Status:", status.status)
    if status.status in ["succeeded", "failed"]:
        break
    time.sleep(10)

fine_tuned_model = status.fine_tuned_model
print("Fine-tuned model:", fine_tuned_model)


## Step 5: Compare Outputs

In [ ]:
query = "Explain RAG"

# 1. Base model
base = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": query}]
)

# 2. Prompt Engineering
prompted = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Explain in a fun, simple way with examples"},
        {"role": "user", "content": query}
    ]
)

# 3. Fine-tuned model
tuned = client.chat.completions.create(
    model=fine_tuned_model,
    messages=[{"role": "user", "content": query}]
)

print("\n--- BASE MODEL ---\n", base.choices[0].message.content)
print("\n--- PROMPT ENGINEERING ---\n", prompted.choices[0].message.content)
print("\n--- FINE-TUNED MODEL ---\n", tuned.choices[0].message.content)


## Key Takeaways

| Method | Strength |
|-------|--------|
| Base Model | General knowledge |
| Prompt Engineering | Quick control |
| Fine-Tuning | Consistent behavior |

### Rule of Thumb:
- Use prompting → fast experiments  
- Use fine-tuning → consistent style  
- Use RAG → external knowledge  
